In [1]:
!pip install cupy-cuda12x  
# or cuda11x depending on CUDA Version

#CuPy (GPU version)

##Two Examples:
 - Sparse × Dense
 - Sparse × Sparse

In [2]:
# Parameters
N, M = 10000, 1000  # Matrix size
density = 0.01
nnz = int(N * N * density)

In [3]:
import cupy as cp
from cupyx.scipy.sparse import csr_matrix
import time


# Generate random sparse matrix A (N x N)
row = cp.random.randint(0, N, nnz)
col = cp.random.randint(0, N, nnz)
data = cp.random.rand(nnz).astype(cp.float32)
sparse_A = csr_matrix((data, (row, col)), shape=(N, N))

# Generate dense matrix B (M x N) // should be N * M
dense_B = cp.random.rand(N, M).astype(cp.float32)

# Sparse × Dense
cp.cuda.Device(0).synchronize()
start = time.time()
result_dense = sparse_A @ dense_B
cp.cuda.Device(0).synchronize()
cupy_sparse_dense_time = time.time() - start
print(f"[CuPy] Sparse × Dense: {cupy_sparse_dense_time:.4f} s")

# Sparse × Sparse
row_B = cp.random.randint(0, N, nnz)
col_B = cp.random.randint(0, M, nnz)
data_B = cp.random.rand(nnz).astype(cp.float32)
sparse_B = csr_matrix((data_B, (row_B, col_B)), shape=(N, M))

cp.cuda.Device(0).synchronize()
start = time.time()
result_sparse = sparse_A @ sparse_B
cp.cuda.Device(0).synchronize()
cupy_sparse_sparse_time = time.time() - start
print(f"[CuPy] Sparse × Sparse: {cupy_sparse_sparse_time:.4f} s")

# Free the memory
cp._default_memory_pool.free_all_blocks()

[CuPy] Sparse × Dense: 0.1696 s
[CuPy] Sparse × Sparse: 0.0682 s


#NumPy + SciPy (CPU version)

In [4]:
import numpy as np
from scipy.sparse import csr_matrix
import time


# Generate random sparse matrix A (N x N)
row = np.random.randint(0, N, nnz)
col = np.random.randint(0, N, nnz)
data = np.random.rand(nnz).astype(np.float32)
sparse_A = csr_matrix((data, (row, col)), shape=(N, N))

# Dense matrix B (N x M)
dense_B = np.random.rand(N, M).astype(np.float32)

# Sparse × Dense
start = time.time()
result_dense = sparse_A @ dense_B
scipy_sparse_dense_time = time.time() - start
print(f"[SciPy] Sparse × Dense: {scipy_sparse_dense_time:.4f} s")

# Sparse × Sparse
row_B = np.random.randint(0, N, nnz)
col_B = np.random.randint(0, M, nnz)
data_B = np.random.rand(nnz).astype(np.float32)
sparse_B = csr_matrix((data_B, (row_B, col_B)), shape=(N, M))

start = time.time()
result_sparse = sparse_A @ sparse_B
scipy_sparse_sparse_time = time.time() - start
print(f"[SciPy] Sparse × Sparse: {scipy_sparse_sparse_time:.4f} s")

[SciPy] Sparse × Dense: 0.1836 s
[SciPy] Sparse × Sparse: 0.2318 s


#Comparison
GPU vs. CPU

In [5]:
print(f"Speedup Sparse × Dense: {scipy_sparse_dense_time / cupy_sparse_dense_time:.2f}x")
print(f"Speedup Sparse × Sparse: {scipy_sparse_sparse_time / cupy_sparse_sparse_time:.2f}x")

Speedup Sparse × Dense: 1.08x
Speedup Sparse × Sparse: 3.40x


# Further Speedup with Block Sparse Matrix

In [6]:
import cupy as cp
from cupyx.scipy.sparse import csr_matrix, bmat
import time

# Block size
block_size = 1000
density = 0.01  # Only 1% of entries are non-zero
N, M = block_size * 2, 10  # Matrix size

# Helper function to create sparse CuPy matrix
def create_sparse_gpu_matrix(shape, density):
    mat = cp.random.rand(*shape)
    mask = cp.random.rand(*shape) < density  # keep only some values
    return csr_matrix(mat * mask)

# Create high-sparsity GPU-based CSR matrices
A = create_sparse_gpu_matrix((block_size, block_size), density)
C = create_sparse_gpu_matrix((block_size, block_size), density)

# Construct block sparse matrix on GPU
sparse_matrix = bmat([[A, None], [None, C]])

# Dense matrix B (M x N) on GPU
dense_matrix = cp.random.rand(M, N, dtype=cp.float32)

# Multiplication w/ block sparsity
cp.cuda.Device(0).synchronize()
start_block = time.time()
result_block =  dense_matrix @ sparse_matrix
cp.cuda.Device(0).synchronize()
time_block = time.time() - start_block

# Multiplication w/o block sparsity (convert to dense CuPy array)
sparse_matrix = sparse_matrix.toarray()
cp.cuda.Device(0).synchronize()
start_no_block = time.time()
result_no_block = dense_matrix @ sparse_matrix
cp.cuda.Device(0).synchronize()
time_no_block = time.time() - start_no_block

# Print results
print("Multiplication time w/ block sparsity:", time_block, "seconds")
print("Multiplication time w/o block sparsity:", time_no_block, "seconds")

# Free the memory
cp._default_memory_pool.free_all_blocks()

Multiplication time w/ block sparsity: 0.0023119449615478516 seconds
Multiplication time w/o block sparsity: 0.011452674865722656 seconds
